In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb
# Install the libraries required to build a RAG application, process PDF documents, generate text embeddings, and store/search vectors using ChromaDB.
# langchain → Framework for building applications using Large Language Models (LLMs).
# langchain-core → Provides the basic components and interfaces of LangChain.
# langchain-community → Provides integrations with external tools, models, and services.
# pypdf → Extracts text from PDF documents.
# pymupdf → Processes PDFs, including extracting text and images.
# sentence-transformers → Converts text into numerical vectors called embeddings.
# chromadb → Stores and searches vector embeddings using similarity.

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc=Document(
    page_content="Hello world",
    metadata={"source":"https://ww.google.com"}
)

In [ ]:
sample_doc

In [ ]:
# Text 
from langchain_community.document_loaders.text import TextLoader  # for load text data.

loader = TextLoader("Data/Python.txt",encoding="utf-8")


In [ ]:
document=loader.load()

In [ ]:
document

In [ ]:
# from langchain_community.document_loaders.pdf import PyPDFLoader # for load pdf data

# pypdf_loader=PyPDFLoader("Data/research2.pdf")

# document=pypdf_loader.load()
# document

In [ ]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pypdf_loader=PyMuPDFLoader("Data/research2.pdf")

# document=pypdf_loader.load()
# document

### Ingestion Pipeline 

In [35]:
import os # For axcess the file from device 
from langchain_community.document_loaders.pdf import PyPDFLoader

In [36]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0 # Track the num of document 
    all_docs=[] # here we store our document 

    for filename in os.listdir(folder_path): # there will be  other file also fro pdf we list this  
        if filename.lower().endswith(".pdf"):

            # complite file path 
            pdf_path=os.path.join(folder_path,filename) # Fro create pdf path 
            loader=PyPDFLoader(pdf_path)
            doc = loader.load()
            all_docs.extend(doc)
            num_docs+=1

    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs)) # For num of pages in pdf 
    return all_docs

            
    

In [37]:
all_pdf_documents=load_all_pdfs() # Hare is our all document 

total pdfs: 2
total pages: 32


In [38]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks 

In [ ]:
# !pip install langchain_text_splitters # For chunks 

In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks_docs = text_splitter.split_documents(documents)
    return chunks_docs

# max num of char each chunks should 


In [40]:
chunks=split_docs(all_pdf_documents) # For doing chunks of all documents from all_pdf_document 

In [7]:
len(chunks)

320

### Embedding 

In [8]:
from sentence_transformers import SentenceTransformer 

In [41]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"): # IN hugging face transformer we use model_name="all-MiniLM-L6-v2" model 
        

        self.model_name=model_name 
        print("model name ...",self.model_name)
        self.model=SentenceTransformer(self.model_name) # loads the pretrained embedding model.
        print("embeddig dimenshion",self.model.get_sentence_embedding_dimension()) # print dimenshion of "all-MiniLM-L6-v2" fro each sentence 

    # generate embedding 
    def generate_embeddings(self,text): #This method accepts text as input.
        embedding=self.model.encode(text,show_progress_bar=True)  #self.model.encode() converts the input text into an embedding
        print("embedding shape:",embedding.shape)
        return embedding 

In [42]:
embedding_manager=EmbeddingManager()

model name ... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embeddig dimenshion 384


C:\Users\aakas\AppData\Local\Temp\ipykernel_38020\4114408751.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embeddig dimenshion",self.model.get_sentence_embedding_dimension()) # print dimenshion of "all-MiniLM-L6-v2" fro each sentence


### Vector Store

In [43]:
import chromadb # Store database 
import uuid # fro create index xx

#the entity that help to  connect with data store 

In [44]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None #the entity that help to  connect with data store 

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(       
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )
#         Now self.collection refers to the actual ChromaDB collection.
#         - self.client connects to ChromaDB.
#         - get_or_create_collection() gets an existing collection or creates one if it doesn't exist.
#         - self.collection holds the returned collection object.

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)): # her we zip to get both information documents and embedding 
            doc_id = f"doc_{uuid.uuid4()}" # unik id for invidual chunks 
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)  #page_content=the actual raw text

            embeddings_list.append(embedding.tolist())

            self.collection.add( #self.collection is like a box inside a database that holds your document data and embeddings.
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [45]:
vector_store=VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [46]:
texts=[doc.page_content for doc in chunks ] # extracts the actual text from every document chunk and stores it in a list called texts.
                                            # for doc in chunks → Takes each chunk one by one.
                                            # doc.page_content → Extracts the text from each chun
embedding=embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embedding)

# chunks → List of document chunks, including text and metadata.
# embedding → Numerical vectors representing each chunk's text.

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embedding shape: (320, 384)
total documents added in vector store= 320
docs in collection: 320


### Retrieval Pipeline 

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

In [61]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # Sementic search 
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # Cosine similarity 
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0] # thsi is a cosine distance ,we will convert this distance with co-sine similarity 

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)): #enumerate() adds an index to each combination, starting from 0 by default
                similarity_score = 1 - distance                            #In your RAG code, i tracks the position of each retrieved documen

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [62]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [63]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 0 documents


[]

### Integrate with llm 

In [ ]:
# !pip install langchain-openai

### Groq

In [ ]:
 # !pip install langchain-groq

In [ ]:
API_Key_GROQ = "API KEY " 



In [65]:
   from langchain_groq import ChatGroq

   llm = ChatGroq(
       groq_api_key=API_Key_GROQ,
       model="openai/gpt-oss-20b",
       temperature=0.1,
       max_tokens=1024
   )

In [66]:
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke(prompt)
    return response.content

In [70]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape: (1, 384)
retrieved 0 documents
we found no relevant context for the given query
**Encoder‑Decoder** is a general neural‑network architecture used to transform one sequence (or set of features) into another.  
It consists of two main parts:

| Component | Role | Typical implementation |
|-----------|------|------------------------|
| **Encoder** | Reads the input (text, image, audio, etc.) and compresses it into a fixed‑size “context” vector (or a sequence of hidden states). | RNN/LSTM/GRU, CNN, Transformer encoder |
| **Decoder** | Generates the output sequence step‑by‑step, conditioned on the context vector and, often, on its own previous outputs. | RNN/LSTM/GRU, Transformer decoder, autoregressive models |

### How it works
1. **Encoding phase**  
   - The encoder processes the input tokens one by one (or in parallel for Transformers).  
   - At each step it updates its hidden state; after the last token it produces a final hidden state (or a set of hidden states).  